# Minimum-width minimizer for one AR(3) process at $k=4$

This notebook generates one stationary AR(3) process with innovations $\varepsilon_t\sim\operatorname{Unif}[-1,1]$, constructs only the $k=p+1=4$ sliding-window point cloud, and minimizes its empirical directional width on the unit sphere.

The optimization uses 12 deterministic starts for projected subgradient descent and retains the run with the smallest width. Because width is invariant under $u\mapsto-u$, the reported sign is aligned with the population AR direction $(1,-\phi_1,-\phi_2,-\phi_3)$.

In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.spatial.distance import pdist
from scipy.stats import mannwhitneyu, spearmanr

AR_CODE_DIR = next(
    (
        candidate.resolve()
        for root in (Path.cwd(), *Path.cwd().parents)
        for candidate in (root / "Code" / "Code_AR_processes", root)
        if (candidate / "functions.py").is_file()
    ),
    None,
)
if AR_CODE_DIR is None:
    raise FileNotFoundError(
        "Could not locate Code/Code_AR_processes from the current directory."
    )
PROJECT_ROOT = AR_CODE_DIR.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Code.Code_AR_processes.functions import (
    batched_geometric_width_curves,
    batched_riemannian_adam_width_curves,
    jones_ar_coefficients,
    projected_subgradient_empirical_width,
    simulate_uniform_ar,
    sliding_window_point_cloud,
)

np.set_printoptions(precision=8, suppress=True)

In [2]:
order = 3
embedding_dimension = order + 1
sample_size = 2_000
burnin = 1_000
coefficient_seed = 20260910
process_seed = 20260911

phi = jones_ar_coefficients(order, rng=coefficient_seed)
ar3_series = simulate_uniform_ar(
    phi,
    sample_size=sample_size,
    burnin=burnin,
    half_width=1.0,
    rng=process_seed,
)
point_cloud_k4 = sliding_window_point_cloud(
    ar3_series,
    embedding_dimension=embedding_dimension,
)
point_cloud_diameter = float(pdist(point_cloud_k4).max())

print(f"AR coefficients: {phi}")
print(f"Embedding dimension: k = {embedding_dimension}")
print(f"Point-cloud shape: {point_cloud_k4.shape}")
print(f"Empirical Lipschitz constant L_T: {point_cloud_diameter:.8f}")

AR coefficients: [-1.0201964  -0.40421235 -0.37447885]
Embedding dimension: k = 4
Point-cloud shape: (1997, 4)
Empirical Lipschitz constant L_T: 36.56167325


In [3]:
number_of_starts = 12
optimization_iterations = 5_000
optimizer_seed = 20260912
initial_step_scale = 0.2
initial_step = initial_step_scale / point_cloud_diameter

# Include the smallest-covariance eigenvector and deterministic random starts.
_, covariance_eigenvectors = np.linalg.eigh(
    np.cov(point_cloud_k4, rowvar=False)
)
initial_direction_generator = np.random.default_rng(optimizer_seed)
initial_directions = [covariance_eigenvectors[:, 0]]
initial_directions.extend(
    initial_direction_generator.normal(size=embedding_dimension)
    for _ in range(number_of_starts - 1)
)

descent_results = [
    projected_subgradient_empirical_width(
        point_cloud_k4,
        iterations=optimization_iterations,
        initial_step=initial_step,
        patience=1_200,
        rng=optimizer_seed + start_index,
        initial_direction=initial_direction,
        step_schedule="geometric",
        geometric_decay=0.997,
    )
    for start_index, initial_direction in enumerate(initial_directions)
]

start_minimum_widths = np.array([
    result["minimum_width"] for result in descent_results
])
best_start = int(np.argmin(start_minimum_widths))
best_result = descent_results[best_start]
estimated_minimizer = best_result["direction"].copy()

# Fix the otherwise arbitrary sign of the minimizing direction.
population_ar_direction = np.concatenate(([1.0], -phi))
population_ar_direction /= np.linalg.norm(population_ar_direction)
if estimated_minimizer @ population_ar_direction < 0.0:
    estimated_minimizer *= -1.0

print(f"Initial step eta_0 = 0.2 / L_T: {initial_step:.10f}")
print(f"Best start: {best_start + 1} of {number_of_starts}")
print(f"Minimum empirical width: {best_result['minimum_width']:.10f}")
print("Minimizing unit direction u_hat:")
print(estimated_minimizer)
print(f"Norm of u_hat: {np.linalg.norm(estimated_minimizer):.10f}")
print("Population AR direction (diagnostic):")
print(population_ar_direction)
print(
    "Absolute cosine similarity: "
    f"{abs(estimated_minimizer @ population_ar_direction):.10f}"
)

Initial step eta_0 = 0.2 / L_T: 0.0054702092
Best start: 8 of 12
Minimum empirical width: 1.3071979716
Minimizing unit direction u_hat:
[0.64295806 0.66166622 0.28492249 0.26004215]
Norm of u_hat: 1.0000000000
Population AR direction (diagnostic):
[0.65310354 0.66629388 0.26399251 0.24457346]
Absolute cosine similarity: 0.9995991548


## Monte Carlo coefficient-recovery experiment

For each of 50 replications, this experiment independently draws stationary AR(3) coefficients with Jones' method, simulates a length-$2{,}000$ process with $\operatorname{Unif}[-1,1]$ innovations, and estimates the minimum-width direction using only the $k=4$ embedding. The geometric schedule is normalized using the empirical Lipschitz constant

$$L_T=\operatorname{diam}(\mathcal P_4)=\max_{s,t}\lVert Y_s^{(4)}-Y_t^{(4)}\rVert,\qquad \eta_t=\frac{0.2}{L_T}(0.997)^t.$$

The coefficient estimate is

$$\widehat\phi_j=-\frac{\widehat u_j}{\widehat u_0},\qquad j=1,2,3.$$

As a classical benchmark, the same point cloud is fitted by $L^\infty$ (Chebyshev) regression. It solves the linear program

$$\min_{\phi,c,r} r \quad\text{subject to}\quad \left|X_t-\sum_{j=1}^3\phi_jX_{t-j}-c\right|\le r \quad\text{for every }t.$$

This minimizes the unnormalized residual range, whereas the proposed method minimizes directional width over the sphere. The LP is therefore used only as a coefficient-estimation benchmark.

The current projected descent is also compared with sphere-aware Adam. Adam uses the same 12 initial directions, 5,000 iterations, and geometric decay. Its hyperparameters are fixed at $\alpha_t=0.2(0.997)^t$, $\beta_1=0.9$, and $\beta_2=0.999$. The first moment is projected into the current tangent space and the scalar second moment is based on the squared tangent-gradient norm. The initial step 0.2 was selected on replications 1--10 from the fixed pilot grid $\{0.02,0.05,0.1,0.2\}$; replications 11--50 form the untouched evaluation set. Both optimizers and the LP receive exactly the same coefficient draws and simulated paths. The reported overall MSE is the mean of all squared coefficient errors.

In [4]:
def chebyshev_ar_fit(point_cloud):
    """Fit AR coefficients by minimizing the maximum absolute residual."""
    response = point_cloud[:, 0]
    lagged_values = point_cloud[:, 1:]
    observation_count = len(response)
    ones = np.ones(observation_count)

    # Variables are (phi_1, phi_2, phi_3, residual_center, radius).
    inequality_matrix = np.vstack((
        np.column_stack((-lagged_values, -ones, -ones)),
        np.column_stack((lagged_values, ones, -ones)),
    ))
    inequality_bounds = np.concatenate((-response, response))
    objective = np.zeros(order + 2)
    objective[-1] = 1.0

    result = linprog(
        objective,
        A_ub=inequality_matrix,
        b_ub=inequality_bounds,
        bounds=[(None, None)] * (order + 1) + [(0.0, None)],
        method="highs",
    )
    if not result.success:
        raise RuntimeError(f"Chebyshev LP failed: {result.message}")
    return result.x[:order], result.x[order], result.x[-1]


replication_count = 50
experiment_seed = 20260913
experiment_rows = []

for replication in range(replication_count):
    process_generator = np.random.default_rng(
        np.random.SeedSequence([experiment_seed, replication, 0])
    )
    true_phi = jones_ar_coefficients(order, rng=process_generator)
    replicated_series = simulate_uniform_ar(
        true_phi,
        sample_size=sample_size,
        burnin=burnin,
        half_width=1.0,
        rng=process_generator,
    )
    replicated_point_cloud = sliding_window_point_cloud(
        replicated_series, embedding_dimension=embedding_dimension
    )

    geometric_start_time = time.perf_counter()
    replicated_lipschitz_constant = float(
        pdist(replicated_point_cloud).max()
    )
    replicated_initial_step = (
        initial_step_scale / replicated_lipschitz_constant
    )
    _, replicated_widths, _, replicated_directions = (
        batched_geometric_width_curves(
            replicated_series,
            [embedding_dimension],
            seed=experiment_seed,
            replication=replication,
            starts=number_of_starts,
            iterations=optimization_iterations,
            initial_step=replicated_initial_step,
            geometric_decay=0.997,
            return_directions=True,
        )
    )
    replicated_minimizer = replicated_directions[0]
    if abs(replicated_minimizer[0]) <= np.finfo(float).eps:
        raise RuntimeError(
            f"Cannot recover coefficients in replication {replication + 1}: "
            "the first minimizer component is numerically zero."
        )
    estimated_phi = (
        -replicated_minimizer[1:] / replicated_minimizer[0]
    )
    true_direction = np.concatenate(([1.0], -true_phi))
    true_direction /= np.linalg.norm(true_direction)
    direction_cosine_similarity = abs(replicated_minimizer @ true_direction)
    geometric_runtime = time.perf_counter() - geometric_start_time
    squared_errors = (estimated_phi - true_phi) ** 2

    adam_start_time = time.perf_counter()
    _, adam_widths, _, adam_directions = (
        batched_riemannian_adam_width_curves(
            replicated_series,
            [embedding_dimension],
            seed=experiment_seed,
            replication=replication,
            starts=number_of_starts,
            iterations=optimization_iterations,
            initial_step=0.2,
            geometric_decay=0.997,
            beta_1=0.9,
            beta_2=0.999,
            return_directions=True,
        )
    )
    adam_minimizer = adam_directions[0]
    adam_phi = -adam_minimizer[1:] / adam_minimizer[0]
    adam_runtime = time.perf_counter() - adam_start_time
    adam_squared_errors = (adam_phi - true_phi) ** 2

    chebyshev_start_time = time.perf_counter()
    chebyshev_phi, chebyshev_center, chebyshev_radius = (
        chebyshev_ar_fit(replicated_point_cloud)
    )
    chebyshev_runtime = time.perf_counter() - chebyshev_start_time
    chebyshev_squared_errors = (chebyshev_phi - true_phi) ** 2

    row = {
        "replication": replication + 1,
        "minimum_width": replicated_widths[0],
        "coefficient_mse": float(np.mean(squared_errors)),
        "lipschitz_constant": replicated_lipschitz_constant,
        "initial_step": replicated_initial_step,
        "abs_u_0": abs(replicated_minimizer[0]),
        "geometric_runtime_seconds": geometric_runtime,
        "direction_cosine_similarity": direction_cosine_similarity,
        "adam_minimum_width": adam_widths[0],
        "adam_coefficient_mse": float(np.mean(adam_squared_errors)),
        "adam_abs_u_0": abs(adam_minimizer[0]),
        "adam_runtime_seconds": adam_runtime,
        "chebyshev_coefficient_mse": float(
            np.mean(chebyshev_squared_errors)
        ),
        "chebyshev_residual_center": chebyshev_center,
        "chebyshev_residual_radius": chebyshev_radius,
        "chebyshev_runtime_seconds": chebyshev_runtime,
    }
    for coefficient_index in range(order):
        suffix = coefficient_index + 1
        row[f"true_phi_{suffix}"] = true_phi[coefficient_index]
        row[f"estimated_phi_{suffix}"] = estimated_phi[coefficient_index]
        row[f"squared_error_phi_{suffix}"] = squared_errors[coefficient_index]
        row[f"adam_phi_{suffix}"] = adam_phi[coefficient_index]
        row[f"adam_squared_error_phi_{suffix}"] = (
            adam_squared_errors[coefficient_index]
        )
        row[f"chebyshev_phi_{suffix}"] = chebyshev_phi[coefficient_index]
        row[f"chebyshev_squared_error_phi_{suffix}"] = (
            chebyshev_squared_errors[coefficient_index]
        )
    experiment_rows.append(row)

    if (replication + 1) % 10 == 0:
        print(f"Completed {replication + 1}/{replication_count} processes")

ar3_mse_results = pd.DataFrame(experiment_rows)

Completed 10/50 processes


Completed 20/50 processes


Completed 30/50 processes


Completed 40/50 processes


Completed 50/50 processes


In [5]:
squared_error_columns = [
    f"squared_error_phi_{index}" for index in range(1, order + 1)
]
chebyshev_squared_error_columns = [
    f"chebyshev_squared_error_phi_{index}"
    for index in range(1, order + 1)
]
adam_squared_error_columns = [
    f"adam_squared_error_phi_{index}"
    for index in range(1, order + 1)
]

method_columns = {
    "Minimum width": {
        "process_mse": "coefficient_mse",
        "squared_errors": squared_error_columns,
        "runtime": "geometric_runtime_seconds",
    },
    "Riemannian Adam": {
        "process_mse": "adam_coefficient_mse",
        "squared_errors": adam_squared_error_columns,
        "runtime": "adam_runtime_seconds",
    },
    "LP Chebyshev": {
        "process_mse": "chebyshev_coefficient_mse",
        "squared_errors": chebyshev_squared_error_columns,
        "runtime": "chebyshev_runtime_seconds",
    },
}
comparison_rows = []
for method, columns in method_columns.items():
    process_mse = ar3_mse_results[columns["process_mse"]]
    comparison_rows.append({
        "method": method,
        "overall_coefficient_mse": float(
            ar3_mse_results[columns["squared_errors"]].to_numpy().mean()
        ),
        "median_process_mse": float(process_mse.median()),
        "maximum_process_mse": float(process_mse.max()),
        "failures_mse_above_1": int((process_mse > 1.0).sum()),
        "mean_runtime_seconds": float(
            ar3_mse_results[columns["runtime"]].mean()
        ),
    })
comparison_summary = pd.DataFrame(comparison_rows).set_index("method")

mse_by_coefficient = pd.DataFrame(
    {
        method: [
            ar3_mse_results[column].mean()
            for column in columns["squared_errors"]
        ]
        for method, columns in method_columns.items()
    },
    index=[f"phi_{index}" for index in range(1, order + 1)],
)
lp_win_count = int(
    (
        ar3_mse_results["chebyshev_coefficient_mse"]
        < ar3_mse_results["coefficient_mse"]
    ).sum()
)

print("Paired comparison on the same 50 processes:")
print(comparison_summary.to_string())
print(f"\nLP Chebyshev has lower process MSE in {lp_win_count}/50 runs.")
adam_win_count = int(
    (
        ar3_mse_results["adam_coefficient_mse"]
        < ar3_mse_results["coefficient_mse"]
    ).sum()
)
adam_lower_width_count = int(
    (
        ar3_mse_results["adam_minimum_width"]
        < ar3_mse_results["minimum_width"]
    ).sum()
)
print(
    f"Adam has lower coefficient MSE in {adam_win_count}/50 runs "
    f"and lower width in {adam_lower_width_count}/50 runs."
)
untouched_evaluation = ar3_mse_results.loc[
    ar3_mse_results["replication"] > 10
]
evaluation_rows = []
for method, mse_column in {
    "Minimum width": "coefficient_mse",
    "Riemannian Adam": "adam_coefficient_mse",
}.items():
    evaluation_mse = untouched_evaluation[mse_column]
    evaluation_rows.append({
        "method": method,
        "overall_mse": float(evaluation_mse.mean()),
        "median_process_mse": float(evaluation_mse.median()),
        "maximum_process_mse": float(evaluation_mse.max()),
        "failures_mse_above_1": int((evaluation_mse > 1.0).sum()),
    })
optimizer_evaluation_summary = pd.DataFrame(evaluation_rows).set_index(
    "method"
)
print("\nUntouched evaluation set (replications 11--50):")
print(optimizer_evaluation_summary.to_string())
print(
    "L_T range: "
    f"{ar3_mse_results['lipschitz_constant'].min():.6f} to "
    f"{ar3_mse_results['lipschitz_constant'].max():.6f}"
)
print(
    "eta_0 range: "
    f"{ar3_mse_results['initial_step'].min():.8f} to "
    f"{ar3_mse_results['initial_step'].max():.8f}"
)
print("MSE by coefficient and method:")
print(mse_by_coefficient.to_string())

ar3_mse_results

Paired comparison on the same 50 processes:
                 overall_coefficient_mse  median_process_mse  maximum_process_mse  failures_mse_above_1  mean_runtime_seconds
method                                                                                                                       
Minimum width                 192.218620            0.000015          8128.038271                     3              0.708303
Riemannian Adam               207.539554            0.000015          8829.080501                     3              0.750793
LP Chebyshev                    0.000007            0.000002             0.000079                     0              0.012411

LP Chebyshev has lower process MSE in 35/50 runs.
Adam has lower coefficient MSE in 27/50 runs and lower width in 39/50 runs.

Untouched evaluation set (replications 11--50):
                 overall_mse  median_process_mse  maximum_process_mse  failures_mse_above_1
method                                                    

,replication,minimum_width,coefficient_mse,lipschitz_constant,initial_step,abs_u_0,geometric_runtime_seconds,direction_cosine_similarity,adam_minimum_width,adam_coefficient_mse,...,adam_squared_error_phi_2,chebyshev_phi_2,chebyshev_squared_error_phi_2,true_phi_3,estimated_phi_3,squared_error_phi_3,adam_phi_3,adam_squared_error_phi_3,chebyshev_phi_3,chebyshev_squared_error_phi_3
0,1,1.177275,8.872332e-05,10.059324,0.019882,0.587373,0.664240,0.999979,1.177265,8.221584e-05,...,4.663095e-05,-0.979864,4.123188e-07,-0.493102,-0.504154,1.221357e-04,-0.503742,1.132149e-04,-0.494075,9.454742e-07
1,2,1.487487,7.709350e-06,12.080760,0.016555,0.745491,0.661298,0.999995,1.487487,7.709344e-06,...,1.254140e-05,0.019370,3.820303e-07,-0.890328,-0.893467,9.853411e-06,-0.893467,9.853421e-06,-0.890377,2.363545e-09
2,3,0.719715,4.438976e-05,74.283141,0.002692,0.359756,0.661000,0.999999,0.719724,3.056015e-05,...,5.682334e-05,-1.650039,3.676569e-06,-0.859893,-0.862644,7.570040e-06,-0.862118,4.952549e-06,-0.858453,2.073109e-06
3,4,1.370186,4.259806e-07,14.216242,0.014068,0.686749,0.659964,1.000000,1.370186,4.259802e-07,...,2.184264e-07,0.318464,2.184269e-07,0.788173,0.789198,1.052060e-06,0.789198,1.052059e-06,0.789198,1.052059e-06
4,5,1.144106,1.197899e-05,41.815389,0.004783,0.572230,0.663391,0.999995,1.144026,7.407555e-06,...,1.370297e-05,-0.147939,2.184595e-06,0.316625,0.314626,3.998336e-06,0.315096,2.340608e-06,0.317533,8.231402e-07
5,6,1.827953,3.814957e-06,6.272393,0.031886,0.916781,0.663889,0.999995,1.827953,3.814974e-06,...,6.952853e-09,0.086738,6.953001e-09,-0.410881,-0.410302,3.345272e-07,-0.410302,3.345355e-07,-0.410302,3.345366e-07
6,7,1.517034,4.439854e-06,9.226494,0.021677,0.759681,0.661550,0.999997,1.517034,4.439859e-06,...,1.170697e-05,-0.250331,1.170698e-05,0.676632,0.677887,1.575305e-06,0.677887,1.575312e-06,0.677887,1.575309e-06
7,8,0.888976,3.111760e-04,29.840818,0.006702,0.440888,0.665966,0.999970,0.888172,4.041162e-05,...,7.772771e-05,-1.072889,2.616984e-06,0.385394,0.394496,8.284325e-05,0.387649,5.083776e-06,0.386416,1.045111e-06
8,9,1.725373,4.924351e-08,8.521748,0.023469,0.863076,0.665511,1.000000,1.725373,4.924485e-08,...,1.721547e-08,0.532004,2.911118e-07,0.219208,0.219340,1.737532e-08,0.219340,1.737606e-08,0.218707,2.508543e-07
9,10,1.261455,6.327601e-05,6.837596,0.029250,0.630251,0.666542,0.999983,1.261455,6.327583e-05,...,9.453400e-05,-0.810406,7.102910e-07,0.406447,0.413900,5.554887e-05,0.413900,5.554873e-05,0.407102,4.296196e-07


## When does minimum width beat LP Chebyshev?

The following is an exploratory paired analysis. The three minimum-width runs with MSE above one are excluded when comparing regular parameter regimes, because they represent convergence to the shifted-direction basin rather than ordinary estimator variation. A nominal win means lower process-level coefficient MSE on the same simulated path. Since several quantities are examined on only 47 regular runs, the $p$-values are descriptive rather than confirmatory.

In [6]:
pattern_results = ar3_mse_results.copy()
pattern_results["minimum_width_wins"] = (
    pattern_results["coefficient_mse"]
    < pattern_results["chebyshev_coefficient_mse"]
)
pattern_results["geometric_failure"] = (
    pattern_results["coefficient_mse"] > 1.0
)
for coefficient_index in range(1, order + 1):
    pattern_results[f"abs_phi_{coefficient_index}"] = abs(
        pattern_results[f"true_phi_{coefficient_index}"]
    )
pattern_results["phi_norm"] = np.sqrt(
    sum(
        pattern_results[f"true_phi_{index}"] ** 2
        for index in range(1, order + 1)
    )
)
pattern_results["true_u_0"] = 1.0 / np.sqrt(
    1.0 + pattern_results["phi_norm"] ** 2
)
pattern_results["direction_error"] = (
    1.0 - pattern_results["direction_cosine_similarity"]
)
pattern_results["log10_relative_advantage"] = np.log10(
    pattern_results["chebyshev_coefficient_mse"]
    / pattern_results["coefficient_mse"]
)

def stationarity_root_margin(row):
    coefficients = np.array([
        row[f"true_phi_{index}"] for index in range(1, order + 1)
    ])
    roots = np.roots(np.concatenate((-coefficients[::-1], [1.0])))
    return float(np.min(abs(roots)) - 1.0)


pattern_results["root_margin"] = pattern_results.apply(
    stationarity_root_margin, axis=1
)
regular_results = pattern_results.loc[
    ~pattern_results["geometric_failure"]
].copy()
winning_results = regular_results.loc[
    regular_results["minimum_width_wins"]
]
losing_results = regular_results.loc[
    ~regular_results["minimum_width_wins"]
]

metric_columns = {
    "|phi_1|": "abs_phi_1",
    "|phi_2|": "abs_phi_2",
    "|phi_3|": "abs_phi_3",
    "||phi||": "phi_norm",
    "true |u_0|": "true_u_0",
    "direction error (1-cosine)": "direction_error",
    "stationarity root margin": "root_margin",
    "L_T": "lipschitz_constant",
}
metric_rows = []
correlation_rows = []
for label, column in metric_columns.items():
    win_values = winning_results[column]
    loss_values = losing_results[column]
    rank_test = mannwhitneyu(
        win_values, loss_values, alternative="two-sided"
    )
    rank_biserial = (
        2.0 * rank_test.statistic / (len(win_values) * len(loss_values))
        - 1.0
    )
    metric_rows.append({
        "metric": label,
        "win_median": float(win_values.median()),
        "loss_median": float(loss_values.median()),
        "rank_biserial_effect": float(rank_biserial),
        "exploratory_p_value": float(rank_test.pvalue),
    })
    rank_correlation = spearmanr(
        regular_results[column],
        regular_results["log10_relative_advantage"],
    )
    correlation_rows.append({
        "metric": label,
        "spearman_rho_with_advantage": float(rank_correlation.statistic),
        "exploratory_p_value": float(rank_correlation.pvalue),
    })

win_pattern_summary = pd.DataFrame(metric_rows).set_index("metric")
advantage_correlations = pd.DataFrame(correlation_rows).set_index("metric")
relative_advantage = (
    pattern_results["chebyshev_coefficient_mse"]
    / pattern_results["coefficient_mse"]
)

print(f"Nominal minimum-width wins: {int(pattern_results['minimum_width_wins'].sum())}/50")
print(f"Wins by more than 10%: {int((relative_advantage > 1.1).sum())}/50")
print(f"Wins by a factor above 2: {int((relative_advantage > 2.0).sum())}/50")
print("\nMedian characteristics of regular wins and losses:")
print(win_pattern_summary.to_string())
print("\nRank correlations with log10(LP MSE / minimum-width MSE):")
print(advantage_correlations.to_string())


Nominal minimum-width wins: 15/50
Wins by more than 10%: 10/50
Wins by a factor above 2: 4/50

Median characteristics of regular wins and losses:
                            win_median  loss_median  rank_biserial_effect  exploratory_p_value
metric                                                                                        
|phi_1|                       0.457863     0.673217             -0.429167             0.019323
|phi_2|                       0.345906     0.477011             -0.266667             0.147287
|phi_3|                       0.563684     0.351348              0.366667             0.045835
||phi||                       0.854127     0.987012             -0.279167             0.129103
true |u_0|                    0.760389     0.712143              0.279167             0.129103
direction error (1-cosine)    0.000001     0.000014             -0.629167             0.000593
stationarity root margin      0.080322     0.112333             -0.304167             0.098010